# device-consistent-construct — worked example 2: Construct an arange index vector on the input's device for gather

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `device-consistent-construct`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Index tensors used by `gather`/advanced indexing must sit on the **same device** as the tensor being indexed. Building them with `t.arange(n, device=x.device)` guarantees the index lives beside the data. Index tensors are integer-typed, so we set `dtype=t.long` explicitly rather than copying `x.dtype` (a float).

## Worked solution

**Goal:** reverse the rows of a 2-D tensor by gathering with an index vector that is built on `x`'s device.

**Step 1 — figure out what the index needs.** To reverse `n` rows we want the index `[n-1, n-2, ..., 0]`. Indices must be `long`, and crucially they must be on `x.device` or `index_select`/`gather` raises a device-mismatch `RuntimeError`.

**Step 2 — build arange on the right device.** `idx = t.arange(n - 1, -1, -1, device=x.device)`. The `device=x.device` kwarg places the counter directly beside the data. We rely on arange's default integer dtype (`long`), which is exactly what indexing wants — here we do NOT thread `x.dtype`, because `x` is a float and a float index is illegal.

**Step 3 — gather.** `x.index_select(0, idx)` selects rows in reversed order. Because `idx` was constructed on `x.device`, this works on CPU or GPU unchanged.

**Why it works:** the rule is *match the device of the operand you interact with*; for an index tensor the dtype rule is different (it must be integer), so we thread device from `x` but leave dtype as arange's integer default.

In [ ]:
def reverse_rows(x):
    n = x.shape[0]
    idx = t.arange(n - 1, -1, -1, device=x.device)
    return x.index_select(0, idx)

t.manual_seed(0)
x = t.randn(4, 3)
out = reverse_rows(x)
print(out.shape, out.device, t.allclose(out, x.flip(0)))